# Adamas University
### Department of Computer Science and Engineering (AI & ML)
## Indian Emigration Q&A Research Project — Demo Notebook

This is an **inference-only demo**. The three fine-tuned `Q4_K_M` GGUF files are already in your Google Drive. This notebook does not download base-model weights, merge adapters, or convert/quantize models.

### Run the demo
1. In Colab, choose **Runtime → Change runtime type → T4 GPU** if a free GPU is offered. If only CPU is available, the demo hides Qwen3 14B to avoid likely memory exhaustion; Mistral and Granite may still run slowly.
2. Run the three code cells below in order and approve the Google Drive access prompt.
3. Wait for the selected model to load, then use the Gradio chat interface. Only one model is loaded at a time.
4. The `gradio.live` link is temporary and works only while this Colab runtime remains connected. Anyone with the link may be able to access the demo; do not enter personal or confidential information.

Expected Drive folder: `My Drive/Research Project/gguf/`

Expected files:
- `Mistral-7B-Finetuned-Q4_K_M.gguf`
- `Granite-3.3-8B-Finetuned-Q4_K_M.gguf`
- `Qwen3-14B-Finetuned-Q4_K_M.gguf`

Model output is a research demonstration, not legal or immigration advice.


In [ ]:
# 1. Mount Drive and verify the three existing GGUF files
from pathlib import Path
from google.colab import drive

GGUF_DIR = Path('/content/drive/MyDrive/Research Project/gguf')
MODEL_FILES = {
    'Mistral 7B': 'Mistral-7B-Finetuned-Q4_K_M.gguf',
    'Granite 3.3 8B': 'Granite-3.3-8B-Finetuned-Q4_K_M.gguf',
    'Qwen3 14B': 'Qwen3-14B-Finetuned-Q4_K_M.gguf',
}

if not GGUF_DIR.is_dir():
    drive.mount('/content/drive')

missing = [name for name in MODEL_FILES.values() if not (GGUF_DIR / name).is_file()]
if missing:
    expected = '\n'.join(f'  - {GGUF_DIR / name}' for name in missing)
    raise FileNotFoundError(
        'Some expected GGUFs were not found in Google Drive. Check the folder and exact filenames:\n'
        + expected
    )

print(f'Found all three fine-tuned GGUFs in {GGUF_DIR}:')
for label, filename in MODEL_FILES.items():
    size_gb = (GGUF_DIR / filename).stat().st_size / 1e9
    print(f'  {label}: {size_gb:.2f} GB')


In [ ]:
%%bash
# 2. Install only the inference UI and llama-server (no training or conversion)
set -euo pipefail

python3 -m pip install -q "gradio==5.49.1" requests
mkdir -p /content/llama-bin /content/llama-extract
curl -fsSL 'https://api.github.com/repos/ggml-org/llama.cpp/releases?per_page=10' -o /tmp/llama-releases.json

pick_asset() {
    python3 - "$1" /tmp/llama-releases.json <<'PY_ASSET'
import json, re, sys
pattern, source = sys.argv[1:]
with open(source, encoding='utf-8') as handle:
    releases = json.load(handle)
for release in releases:
    for asset in release.get('assets', []):
        if re.search(pattern, asset['name']):
            print(asset['browser_download_url'])
            raise SystemExit(0)
raise SystemExit(f'No llama.cpp release asset matched: {pattern}')
PY_ASSET
}

download_and_extract() {
    local pattern="$1"
    local archive="$2"
    local url
    url=$(pick_asset "$pattern")
    echo "Downloading $(basename "$url")"
    wget -q --show-progress "$url" -O "$archive"
    tar -xzf "$archive" -C /content/llama-extract
}

BACKEND=cpu
if command -v nvidia-smi >/dev/null 2>&1 && nvidia-smi >/dev/null 2>&1; then
    BACKEND=cuda
    download_and_extract '^llama-.*-bin-ubuntu-cuda-12\.8-x64\.tar\.gz$' /tmp/llama-main.tar.gz
    download_and_extract '^cudart-llama-.*-bin-ubuntu-cuda-12\.8-x64\.tar\.gz$' /tmp/llama-cudart.tar.gz
else
    download_and_extract '^llama-.*-bin-ubuntu-x64\.tar\.gz$' /tmp/llama-main.tar.gz
fi

find /content/llama-extract -type f -exec cp -f {} /content/llama-bin/ \;
chmod +x /content/llama-bin/llama-* 2>/dev/null || true
cp -a /content/llama-bin/*.so* /usr/lib/ 2>/dev/null || true
ldconfig
export LD_LIBRARY_PATH=/content/llama-bin:${LD_LIBRARY_PATH:-}

if ! /content/llama-bin/llama-server --version >/tmp/llama-version.txt 2>&1; then
    if [ "$BACKEND" != "cuda" ]; then
        cat /tmp/llama-version.txt
        exit 1
    fi
    echo "CUDA binary is incompatible with this runtime; falling back to CPU."
    BACKEND=cpu
    rm -rf /content/llama-bin /content/llama-extract
    mkdir -p /content/llama-bin /content/llama-extract
    download_and_extract '^llama-.*-bin-ubuntu-x64\.tar\.gz$' /tmp/llama-main.tar.gz
    find /content/llama-extract -type f -exec cp -f {} /content/llama-bin/ \;
    chmod +x /content/llama-bin/llama-* 2>/dev/null || true
    cp -a /content/llama-bin/*.so* /usr/lib/ 2>/dev/null || true
    ldconfig
    /content/llama-bin/llama-server --version >/tmp/llama-version.txt
fi

echo "$BACKEND" > /content/llama-bin/backend.txt
cat /tmp/llama-version.txt
printf 'Inference backend: %s\n' "$BACKEND"


In [ ]:
# 3. Download and start the version-pinned demo launcher
from pathlib import Path
import re
import subprocess
import sys
import time
from urllib.request import urlopen

DEMO_SCRIPT_REF = "17ea187488bc106e9825d4d4d9b3a0c67275f614"
DEMO_SCRIPT_URL = (
    "https://raw.githubusercontent.com/veerpast/indian-emigration-llm/"
    f"{DEMO_SCRIPT_REF}/scripts/launch_colab_demo.py"
)
DEMO_SCRIPT_PATH = Path("/content/launch_colab_demo.py")
DEMO_LOG_PATH = Path("/content/launch_colab_demo.log")

with urlopen(DEMO_SCRIPT_URL, timeout=30) as response:
    script_source = response.read()
if not script_source.startswith(b"#!/usr/bin/env python3"):
    raise RuntimeError("The downloaded demo launcher did not match the expected script.")
DEMO_SCRIPT_PATH.write_bytes(script_source)

if "DEMO_PROCESS" not in globals() or DEMO_PROCESS.poll() is not None:
    with DEMO_LOG_PATH.open("w", encoding="utf-8") as log_handle:
        DEMO_PROCESS = subprocess.Popen(
            [sys.executable, "-u", str(DEMO_SCRIPT_PATH)],
            stdout=log_handle,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    print(f"Starting demo helper from commit {DEMO_SCRIPT_REF}...")
else:
    print("Demo process is already running; checking its startup log.")

for _ in range(30):
    if DEMO_PROCESS.poll() is not None:
        log_text = DEMO_LOG_PATH.read_text(encoding="utf-8", errors="replace")
        raise RuntimeError(f"Demo server exited during startup:\n{log_text[-5000:]}")
    log_text = DEMO_LOG_PATH.read_text(encoding="utf-8", errors="replace")
    share_url = re.search(r"https://[A-Za-z0-9.-]+\.gradio\.live\b", log_text)
    if share_url:
        print(f"Demo is ready: {share_url.group(0)}")
        break
    time.sleep(2)
else:
    print("The server is still starting. Recent launcher output:")
    print(log_text[-3000:] or "No launcher output yet.")
    print(f"Rerun this cell to check startup again. Log: {DEMO_LOG_PATH}")
